In [ ]:
pip install opencv-python mediapipe numpy yt-dlp

   ---------------------------------------- 0.0/3.3 MB ? eta -:--:--
   ---------------------------------------- 3.3/3.3 MB 38.7 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


 Download Clip to Specified Folder

In [47]:
import yt_dlp, pathlib, os, logging

logging.basicConfig(level=logging.INFO)

def download_clip(
    url: str,
    start_sec: int | None = None,
    end_sec: int | None = None,
    output_dir: pathlib.Path
) -> pathlib.Path:
    """
    Downloads a clip from the given URL.
    - If `start_sec`/`end_sec` are specified, downloads that time range.
    - If they are `None`, downloads the whole video.
    Returns the MP4 file path or None if failed.
    """
    output_dir.mkdir(parents=True, exist_ok=True)

    try:
        ydl = yt_dlp.YoutubeDL({
            'format': 'mp4',
            # Only set download_sections if both start_sec and end_sec are given
            **({'download_sections': f'{start_sec}-{end_sec}' if start_sec is not None and end_sec is not None else {} })
        })
        result = ydl.download([url])
        if not result:
            logging.warning(f"No output file for {url}.")
            return None
        return pathlib.Path(result[0]).resolve()
    except yt_dlp.DownloadError as e:
        logging.error(f"Download error: {e}")
        return None


SyntaxError: non-default argument follows default argument (1434461305.py, line 9)

Load video frames

In [ ]:
import cv2

def load_video(path: pathlib.Path) -> List[Tuple[int, int]]:
    """Return a list of frames from the video."""
    cap = cv2.VideoCapture(str(path))
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    cap.release()
    return frames


C. Pose Estimation with MediaPipe

In [31]:
import mediapipe as mp
import numpy as np
from typing import List, Tuple

def pose_estimation(frames: List[Tuple[int, int]]) -> List[List[Tuple[float, float]]]:
    """For each frame, return a list of (x, y) coordinates for all landmarks."""
    mp_pose = mp.solutions.pose
    with mp_pose.Pose(
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5) as pose:
        skeletons = []
        for frame in frames:
            results = pose.process(frame)
            if not results.pose_landmarks:
                continue  # skip if no detection
            landmarks = [(lm.x, lm.y) for lm in results.pose_landmarks.landmark]
            skeletons.append(landmarks)
    return skeletons


D Compute Similarity Similarity & Grade

In [34]:
def compute_similarity(skeleton1: List[Tuple[float, float]],
                       skeleton2: List[Tuple[float, float]]) -> float:
    """
    Euclidean distance per joint, then average.  
    Score 0–100 (higher = more similar).
    """
    if len(skeleton1) != len(skeleton2):
        raise ValueError("Skeleton lengths differ")

    distances = [
        np.linalg.norm(np.array(p1) - np.array(p2))
        for p1, p2 in zip(skeleton1, skeleton2)
    ]
    avg_dist = np.mean(distances)

    # Normalization: max possible distance depends on frame resolution
    # For simplicity, assume 0–100 range
    similarity_score = 100 * (1 - avg_dist / max_possible_distance)  # adjust as needed
    return similarity_score


Grade & Suggestion

In [35]:
def grade_student(score: float, threshold=80) -> str:
    """Simple grading: Excellent >80, Good ≥40, Needs Improvement <40."""
    if score >= threshold:
        return "Excellent"
    elif score >= threshold / 2:
        return "Good"
    else:
        return "Needs Improvement"

def suggest_improvement(
    skeleton1: List[Tuple[float, float]],
    skeleton2: List[Tuple[float, float]]) -> str:
    """Identify the joint with largest deviation and give a suggestion."""
    differences = np.array(skeleton1) - np.array(skeleton2)
    max_diff_index = np.argmax(np.linalg.norm(differences, axis=1))
    mp_pose_landmark_names = [
        "nose", "left_eye", "right_eye",
        "left_ear", "right_ear", "mouth_left",
        "mouth_right", "left_shoulder", "right_shoulder",
        "left_elbow", "right_elbow", "left_wrist",
        "right_wrist", "left_pelvis", "right_pelvis",
        "left_knee", "right_knee", "left_ankle",
        "right_ankle"
    ]
    joint_name = mp_pose_landmark_names[max_diff_index]
    suggestion = f"Improve your {joint_name} posture."
    return suggestion


In [44]:
import pathlib, os

def main(
    instructor_url: str,
    instructor_start_sec: int,
    instructor_end_sec: int,
    student_url: str,
    student_start_sec: int,
    student_end_sec: int,
    output_dir_path: str = r"C:\Users\arrun\OneDrive\Desktop\Computer Vision\Temp Taekwando project",
):
    # Convert output path to pathlib
    output_dir = pathlib.Path(output_dir_path)

    # Download instructor clip
    instructor_path = download_clip(
        instructor_url,
        instructor_start_sec,
        instructor_end_sec,
        output_dir
    )
    if not instructor_path:
        raise FileNotFoundError(f"Failed to download instructor clip from {instructor_url}.")

    # Download student clip
    student_path = download_clip(
        student_url,
        student_start_sec,
        student_end_sec,
        output_dir
    )
    if not student_path:
        raise FileNotFoundError(f"Failed to download student clip from {student_url}.")

    # Load videos
    instructor_frames = load_video(instructor_path)
    student_frames   = load_video(student_path)

    # Pose estimation
    instructor_skeletons = pose_estimation(instructor_frames)
    student_skeletons   = pose_estimation(student_frames)

    # Ensure equal number of frames (or use first N common)
    min_frames = min(len(instructor_skeletons), len(student_skeletons))
    if min_frames == 0:
        logging.info("No valid skeletons detected. Exiting.")
        return

    instructor_skeletons = instructor_skeletons[:min_frames]
    student_skeletons   = student_skeletons[:min_frames]

    # Compute similarity per frame
    similarities = [
        compute_similarity(inst, st)
        for inst, st in zip(instructor_skeletons, student_skeletons)
    ]

    avg_score = np.mean(similarities)

    grade = grade_student(avg_score)
    suggestion = suggest_improvement(
        instructor_skeletons[0],  # use first frame as reference
        student_skeletons[0]
    )

    print("\n--- Taekwando Assessment ---")
    print(f"Average similarity score: {avg_score:.2f} / 100")
    print(f"Grade: {grade}")
    print(f"Suggested improvement: {suggestion}")

if __name__ == "__main__":
    # Example URLs & times
    instructor_video_url = "https://www.youtube.com/watch?v=Ot-rBhiIUKs"
    instructor_start_sec = 80   # start at 80 seconds
    instructor_end_sec   = 120  # end at 120 seconds

    student_video_url = "https://www.youtube.com/watch?v=etgxusKS0Do"
    student_start_sec = 153  # 2:53 (153 sec)
    student_end_sec   = 208  # 3:28 (208 sec)

    main(
        instructor_video_url,
        instructor_start_sec,
        instructor_end_sec,
        student_video_url,
        student_start_sec,
        student_end_sec
    )


[youtube] Extracting URL: https://www.youtube.com/watch?v=Ot-rBhiIUKs
[youtube] Ot-rBhiIUKs: Downloading webpage
[youtube] Ot-rBhiIUKs: Downloading webpage


[youtube] Ot-rBhiIUKs: Downloading android sdkless player API JSON
[youtube] Ot-rBhiIUKs: Downloading web safari player API JSON
[youtube] Ot-rBhiIUKs: Downloading web safari player API JSON


[youtube] Ot-rBhiIUKs: Downloading m3u8 information


[info] Ot-rBhiIUKs: Downloading 1 format(s): 96
[download] Taekwondo Basic Form 1 - Full Tutorial [Ot-rBhiIUKs].mp4 has already been downloaded
[download] 100% of    9.85MiB
[download] Taekwondo Basic Form 1 - Full Tutorial [Ot-rBhiIUKs].mp4 has already been downloaded
[download] 100% of    9.85MiB


FileNotFoundError: Failed to download instructor clip from https://www.youtube.com/watch?v=Ot-rBhiIUKs.

In [46]:
!python taekwando_assessment.py


Traceback (most recent call last):
  File "c:\Users\arrun\OneDrive\Desktop\Computer Vision\Temp Taekwando project\taekwando_assessment.py", line 99, in <module>
    "execution_count": null,
                       ^^^^
NameError: name 'null' is not defined


In [38]:
!yt-dlp "https://www.youtube.com/watch?v=Ot-rBhiIUKs" -f bestvideo[ext=mp4] --download-sections 80-120

[youtube] Extracting URL: https://www.youtube.com/watch?v=Ot-rBhiIUKs
[youtube] Ot-rBhiIUKs: Downloading webpage
[youtube] Ot-rBhiIUKs: Downloading android sdkless player API JSON
[youtube] Ot-rBhiIUKs: Downloading web safari player API JSON
[youtube] Ot-rBhiIUKs: Downloading m3u8 information
[info] Ot-rBhiIUKs: Cannot match chapters since chapter information is unavailable


In [39]:
!yt-dlp "https://www.youtube.com/watch?v=Ot-rBhiIUKs" -f mp4 \
      --download-sections 80-120

[youtube] Extracting URL: https://www.youtube.com/watch?v=Ot-rBhiIUKs
[youtube] Ot-rBhiIUKs: Downloading webpage
[youtube] Ot-rBhiIUKs: Downloading android sdkless player API JSON
[youtube] Ot-rBhiIUKs: Downloading web safari player API JSON
[youtube] Ot-rBhiIUKs: Downloading m3u8 information
[info] Ot-rBhiIUKs: Cannot match chapters since chapter information is unavailable


         Pre-merged mp4 formats are not available from all sites, or may only be available in lower quality.
         To prioritize the best h264 video and aac audio in an mp4 container, use "-t mp4" instead.
         If you know what you are doing and want a pre-merged mp4 format, use "-f b[ext=mp4]" instead to suppress this warning


In [40]:
!yt-dlp "https://www.youtube.com/watch?v=Ot-rBhiIUKs" -f bestvideo[ext=mp4]

[youtube] Extracting URL: https://www.youtube.com/watch?v=Ot-rBhiIUKs
[youtube] Ot-rBhiIUKs: Downloading webpage
[youtube] Ot-rBhiIUKs: Downloading android sdkless player API JSON
[youtube] Ot-rBhiIUKs: Downloading web safari player API JSON
[youtube] Ot-rBhiIUKs: Downloading m3u8 information
[info] Ot-rBhiIUKs: Downloading 1 format(s): 399
[download] Taekwondo Basic Form 1 - Full Tutorial [Ot-rBhiIUKs].mp4 has already been downloaded

[download] 100% of    9.85MiB


In [19]:
download_clip("https://www.youtube.com/watch?v=Ot-rBhiIUKs", 80, 120)


'Ot-rBhiIUKs.mp4'

In [20]:
import os

# Get the current working directory
cwd = os.getcwd()
print(f"Current working directory: {cwd}")

# List all files in the current directory
files = os.listdir(cwd)
print("\nFiles in current directory:")
for file in files:
    if file.endswith('.mp4'):
        file_path = os.path.join(cwd, file)
        file_size = os.path.getsize(file_path) / (1024 * 1024)  # Convert to MB
        print(f"  {file} ({file_size:.2f} MB)")

Current working directory: c:\Users\arrun\OneDrive\Desktop\Computer Vision\Temp Taekwando project

Files in current directory:
